In [6]:
from langchain_text_splitters import MarkdownHeaderTextSplitter


def prepare_docs():
    # =============1.准备文档================
    with open('resources/评估.md', 'r', encoding='utf-8') as f:
        doc = f.read()

    # =============2.切分文档================
    # 切分依据，这里是按照三级标题
    headers_to_split_on = [
        ("#", "h1"),
        ("##", "h2"),
        ("###", "h3"),
    ]

    # 创建切分器
    markdown_splitter = MarkdownHeaderTextSplitter(
        headers_to_split_on,
        strip_headers=False  # 在正文中是否去除标题，False，不去除
    )

    # 切分文档, 返回的是 list[Document]
    docs = markdown_splitter.split_text(doc)

    return docs

docs = prepare_docs()

In [7]:
docs[0]

Document(metadata={'h1': '第一章 教育概述', 'h2': '第一节 中外教育家及其教育思想', 'h3': '（一）教育学萌芽时期代表作'}, page_content='# 第一章 教育概述  \n## 第一节 中外教育家及其教育思想  \n### （一）教育学萌芽时期代表作  \n国内：《学记》，世界最早，成文于战国末期，作者是乐正克。  \n国外：昆体良《雄辩术原理》/《论演说家的教育》。')

In [8]:
import sys
from pathlib import Path

# Notebook 的工作目录通常是当前文件所在目录，向上查找服务根目录。
service_root = next(
    (
        path
        for path in (Path.cwd(), *Path.cwd().parents)
        if (path / "pyproject.toml").is_file()
        and (path / "app" / "core" / "config.py").is_file()
    ),
    None,
)
if service_root is None:
    raise RuntimeError("未找到 agent-service 根目录，请从该项目或 notebooks 子目录运行。")

if str(service_root) not in sys.path:
    sys.path.insert(0, str(service_root))


In [3]:
from langchain_ollama import OllamaEmbeddings

# 1.初始化向量模型，可以把文档转稠密向量
embeddings = OllamaEmbeddings(
    model="qwen3-embedding:0.6b",  # 性价比高的模型
    dimensions=1024,  # 可选：减少维度以节省存储
    base_url="http://10.40.0.31:11434",  # 可选：指定 Ollama 服务器地址
)

In [9]:
from app.core.config import settings
from pymilvus import MilvusClient

# 初始化客户端
client = MilvusClient(uri=settings.rag.milvus_url)

# Collection名称
collection_name = "my_collection_1"

In [8]:
# 创建文档的 Schema
from pymilvus import DataType, Function, FunctionType


def create_collection():
    # 1.schema
    schema = client.create_schema(
        auto_id=False,  # 不自动生成id
        enable_dynamic_field=False,  # 是否允许动态字段
    )

    # 1.1.主键字段
    schema.add_field(field_name="id", datatype=DataType.INT64, is_primary=True)
    # 1.2.稠密向量字段
    schema.add_field(field_name="dense", datatype=DataType.FLOAT_VECTOR, dim=1024)
    # 1.3.稀疏向量字段
    schema.add_field(field_name="sparse", datatype=DataType.SPARSE_FLOAT_VECTOR)
    # 1.4.普通元数据字段，用来做数据过滤
    schema.add_field(field_name="h1", datatype=DataType.VARCHAR, max_length=128)
    schema.add_field(field_name="h2", datatype=DataType.VARCHAR, max_length=128)
    schema.add_field(field_name="h3", datatype=DataType.VARCHAR, max_length=128)
    # 1.5.原始文档内容，需要被向量化
    schema.add_field(
        field_name="content",
        datatype=DataType.VARCHAR,
        max_length=2048,
        enable_analyzer=True,  # 允许对该字段分词，转BM25稀疏向量
        analyzer_params={"type": "chinese"}  # 指定分词方式，这里选择中文分词
    )
    # 1.6.添加 BM25 函数，可以自动把原始文档转 BM25 向量存储
    bm25_function = Function(
        name="text_bm25_emb",  # Function name
        input_field_names=["content"],  # 包含原始文档数据的字段的名称
        output_field_names=["sparse"],  # 稀疏向量字段的名称
        function_type=FunctionType.BM25,  # Set to `BM25`
    )
    schema.add_function(bm25_function)

    # 2.索引
    index_params = client.prepare_index_params()
    # 2.1.稠密向量索引
    index_params.add_index(
        field_name="dense",  # 字段名
        index_type="AUTOINDEX",  # 稠密向量索引方式
        metric_type="COSINE"  # 相似度计算方式，COSINE、IP
    )
    # 2.2.稀疏向量索引
    index_params.add_index(
        field_name="sparse",  # 字段名
        index_type="SPARSE_INVERTED_INDEX",  # 稀疏向量，采用倒排索引
        metric_type="BM25",  # 相似度计算方式，支持 BM25、IP
    )
    # 2.3.普通字段索引
    index_params.add_index(field_name="h1", index_type="AUTOINDEX")
    index_params.add_index(field_name="h2", index_type="AUTOINDEX")
    index_params.add_index(field_name="h3", index_type="AUTOINDEX")

    # 3.创建collection
    client.create_collection(
        collection_name=collection_name,
        schema=schema,
        index_params=index_params
    )

create_collection()

In [9]:
# 查看 Collection
res = client.list_collections()
print(res)

['my_collection_1']


In [10]:
# Entity 的操作
# 1.insert
# 稠密向量嵌入
vectors = embeddings.embed_documents([doc.page_content for doc in docs])

# milvus 接收的是 dict，先进行数据转换
data = [
    {
        "id": i+1,
        "content": doc.page_content,
        "h1": doc.metadata.get("h1", ""),
        "h2": doc.metadata.get("h2", ""),
        "h3": doc.metadata.get("h3", ""),
        "dense": vectors[i],  # 稠密向量字段
    }
    for i, doc in enumerate(docs)
]

# 插入
res = client.insert(collection_name, data)
print(res)

{'insert_count': 21, 'ids': [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21]}


In [12]:
# 2.upsert
# 如果 id 存在等同于更新；如果 id 不存在等同于新增

# 准备两条数据
# id 为 1 的数据，修改其 h2 字段值，模拟更新
data1 = data[0]
data1['h2'] = "测试一下"

 # id 为 2 的数据，将其 id 修改为 88，模拟新增
data2 = data[1]
data2['id'] = 88

# 测试 upsert
res = client.upsert(collection_name, [data1, data2])
print(res)# {'upsert_count': 2, 'ids': [1, 88]}

{'upsert_count': 2, 'ids': [1, 88]}


In [13]:
# 3.delete
res = client.delete(collection_name, ids=[88])
print(res)

{'delete_count': 1}


In [1]:
# search
# 1.稠密向量检索
def dense_search(query: str):
    # 问题向量化
    query_vector = embeddings.embed_query(query)

    # 搜索
    res = client.search(
        collection_name=collection_name,    # Collection名称
        anns_field="dense",                 # 检索字段名（稠密）
        data=[query_vector],                # 问题向量（稠密）
        limit=3,                            # 查询文档最大数量
        search_params={"metric_type": "COSINE"},  # 向量相似度算法，必须和索引时一致
        output_fields=["content", "h1", "h2", "h3"]  # 要返回的字段名，默认只返回id和score
    )

    #  遍历结果
    for hits in res:
        for hit in hits:
            print(hit)

In [10]:
# 1.1.语义相似度查询
dense_search("教育的负面作用是什么？")

{'id': 15, 'distance': 0.6178877949714661, 'entity': {'h2': '第一节 教育的功能', 'h3': '（一）个体发展功能和社会发展功能', 'content': '# 第二章 教育基本原理  \n## 第一节 教育的功能  \n### （一）个体发展功能和社会发展功能  \n教育的正向功能（积极功能）指教育有助于社会进步和个体发展的积极影响和作用。  \n教育的负向功能（消极功能）指阻碍社会进步和个体发展的消极影响和作用。  \n教育的显性功能是指教育活动依照教育目的，在实际运行中所出现的与之相吻合的结果。  \n教育的隐性功能指伴随显性功能所出现的非预期性的功能。', 'h1': '第二章 教育基本原理'}}
{'id': 21, 'distance': 0.5609424114227295, 'entity': {'h2': '第二节 教育和社会的关系', 'h3': '（六）生产力与教育的关系', 'content': '### （六）生产力与教育的关系  \n生产力对其它一切因素都起着决定的作用，是决定教育性质的根本因素。  \n生产力对教育的主要作用表现为：生产力发展水平决定着教育事业发展的规模和速度；生产力发展水平制约着人才培养的规格与教育结构；生产力的发展促进教育内容、教学方法和教学组织形式的发展与改革。  \n教育对生产力的作用表现为：教育再生产劳动力；教育是科学知识与技术发展的重要手段。', 'h1': '第二章 教育基本原理'}}
{'id': 18, 'distance': 0.5538069605827332, 'entity': {'h2': '第二节 教育和社会的关系', 'h3': '（三）文化与教育的关系', 'content': '### （三）文化与教育的关系  \n文化对教育的影响：文化影响着教育的价值取向；文化影响着教育的课程内容；文化影响着育人模式；文化影响着教育教学方法的使用。  \n教育对文化的作用表现在：教育传递和保存文化；教育传播和交流文化；教育选择和提升文化；教育创造和更新文化。', 'h1': '第二章 教育基本原理'}}


In [11]:
# 1.2.专有名词查询
dense_search("给教师的建议是谁写的")

{'id': 1, 'distance': 0.4876022934913635, 'entity': {'h2': '测试一下', 'h3': '（一）教育学萌芽时期代表作', 'content': '# 第一章 教育概述  \n## 第一节 中外教育家及其教育思想  \n### （一）教育学萌芽时期代表作  \n国内：《学记》，世界最早，成文于战国末期，作者是乐正克。  \n国外：昆体良《雄辩术原理》/《论演说家的教育》。', 'h1': '第一章 教育概述'}}
{'id': 6, 'distance': 0.4536695182323456, 'entity': {'h2': '第一节 中外教育家及其教育思想', 'h3': '（六）希腊三贤', 'content': '### （六）希腊三贤  \n**苏格拉底**：产婆术（谈话法），分三步：讽刺—定义—助产术。国外启发式教育第一人。  \n**柏拉图**：《理想国》提出普及教育的主张，教育目的是培养哲学王。  \n**亚里士多德**：提出自由教育，提倡对学生进行和谐全面发展的教育，灵魂说。', 'h1': '第一章 教育概述'}}
{'id': 8, 'distance': 0.44780048727989197, 'entity': {'h2': '第一节 中外教育家及其教育思想', 'h3': '（八）教育学分化时期代表人物及主要思想', 'content': '### （八）教育学分化时期代表人物及主要思想  \n**马卡连柯**：著有《教育诗》，提出集体主义教育。  \n**克鲁普斯卡娅**：著有《国民教育与民主制度》，是最早以马克思主义为基础探讨教育问题的教育家。  \n**杨贤江**：著有《新教育大纲》，是中国第一部以马克思主义为指导的教育学著作。  \n**凯洛夫**：著有《教育学》，是世界第一部马克思主义的教育学著作。  \n**布鲁姆**：著有《教学目标分类学》，提出了掌握学习理论：所有学生都能学好；目标分为认知、情感、动作技能。  \n**布鲁纳**：著有《教学过程》，提出了结构主义教学理论，倡导发现式学习。  \n**瓦·根舍因**：著有《范例教学理论》，与布鲁纳和赞可夫被认为课程现代化的三大代表人物。  \n**赞可夫**：著有《教学与发展》，提出了发展性教学理论的五原则：高难度、

In [12]:
# 2.定义稀疏检索函数
def sparse_search(query: str):
    # 直接检索，内置的hm25函数会自动把query转向量
    res = client.search(
        collection_name=collection_name,  # Collection名称
        data=[query],
        anns_field="sparse",  # 检索字段名（稀疏）
        limit=3,  # 查询文档最大数量
        output_fields=["content", "h2"]  # 要返回的字段名
    )
    
    # 打印结果
    for hits in res:
        for hit in hits:
            print(hit)

In [13]:
# 测试模糊的语义搜索，稀疏检索不太擅长
sparse_search("教育的负面作用是什么")

{'id': 21, 'distance': 1.9839448928833008, 'entity': {'content': '### （六）生产力与教育的关系  \n生产力对其它一切因素都起着决定的作用，是决定教育性质的根本因素。  \n生产力对教育的主要作用表现为：生产力发展水平决定着教育事业发展的规模和速度；生产力发展水平制约着人才培养的规格与教育结构；生产力的发展促进教育内容、教学方法和教学组织形式的发展与改革。  \n教育对生产力的作用表现为：教育再生产劳动力；教育是科学知识与技术发展的重要手段。', 'h2': '第二节 教育和社会的关系'}}
{'id': 20, 'distance': 1.9515604972839355, 'entity': {'content': '### （五）人口与教育的关系  \n人口对教育的作用主要表现在：人口的数量和增长是决定教育事业的规模、速度的一个重要因素；人口的质量影响教育的质量；人口的结构影响着教育的发展；人口流动影响着教育的发展。  \n教育对人口的作用主要表现在：控制人口的数量；提高人口的质量；改善人口结构；调整人才构成与流动等作用。', 'h2': '第二节 教育和社会的关系'}}
{'id': 5, 'distance': 1.9163810014724731, 'entity': {'content': '### （五）荀子主要思想  \n**人性论**：性恶论，化性起伪，善德是后天习得的，重视教育的作用。  \n**教学原则**：学以致用，锲而不舍。', 'h2': '第一节 中外教育家及其教育思想'}}


In [14]:
# 专有名称查询：
sparse_search("给教师的建议是谁写的")

{'id': 8, 'distance': 5.255347728729248, 'entity': {'h2': '第一节 中外教育家及其教育思想', 'content': '### （八）教育学分化时期代表人物及主要思想  \n**马卡连柯**：著有《教育诗》，提出集体主义教育。  \n**克鲁普斯卡娅**：著有《国民教育与民主制度》，是最早以马克思主义为基础探讨教育问题的教育家。  \n**杨贤江**：著有《新教育大纲》，是中国第一部以马克思主义为指导的教育学著作。  \n**凯洛夫**：著有《教育学》，是世界第一部马克思主义的教育学著作。  \n**布鲁姆**：著有《教学目标分类学》，提出了掌握学习理论：所有学生都能学好；目标分为认知、情感、动作技能。  \n**布鲁纳**：著有《教学过程》，提出了结构主义教学理论，倡导发现式学习。  \n**瓦·根舍因**：著有《范例教学理论》，与布鲁纳和赞可夫被认为课程现代化的三大代表人物。  \n**赞可夫**：著有《教学与发展》，提出了发展性教学理论的五原则：高难度、高速度、理论知识起主导作用、理解学习过程、所有学生包括差生都得到发展的原则。  \n**苏霍姆林斯基**：著有《给教师的建议》《把整个心灵献给孩子》《帕夫雷什中学》，提出个性全面和谐发展的教育思想，他的著作被称为“活的教育学”。  \n**巴班斯基**：著有《教学过程最优化》《教学教育过程最优化》，把现代控制论、系统论观点用于教学论研究，提出教学过程最优化的理论。  \n**皮亚杰**：提出儿童认知发展阶段论、道德发展阶段论，以及儿童与环境互相作用的两个过程：同化、顺应。  \n**维果斯基**：提出心理学的“文化—历史”发展理论；最近发展区理论，教学必须走在发展的前面；支架式教学；“内化”学说。'}}
{'id': 7, 'distance': 2.7844624519348145, 'entity': {'h2': '第一节 中外教育家及其教育思想', 'content': '### （七）教育学创立时期代表人物及主要思想  \n**昆体良**：西方第一个专门论述教育问题的教育家。  \n**培根**：在《论科学的价值和发展》中首次把“教育学”作为一门独立的科学确立下来。  \n**夸美纽斯**（教育学之父、教育史上哥白尼）：代表作《大教学论》是教育

In [15]:
# RRF 混合检索
from pymilvus import AnnSearchRequest


def hybrid_search(query: str, ranker):
    # 1.创建稠密、稀疏请求
    # 稠密
    query_dense_vector = embeddings.embed_query(query)
    dense_request = AnnSearchRequest(
        data=[query_dense_vector],
        anns_field="dense",
        param={"metric_type": "COSINE"},
        limit=3,
    )
    # 稀疏
    sparse_request = AnnSearchRequest(
        data=[query],
        anns_field="sparse",
        param={"metric_type": "BM25"},
        limit=3,
    )
    # 混合在一起
    reqs = [dense_request, sparse_request]

    # 2.设定融合策略，发送请求
    res = client.hybrid_search(
        collection_name=collection_name,
        reqs=reqs,
        ranker=ranker,
        limit=2,
        output_fields=['content', 'h2']
    )
    for hits in res:
        print("TopK results:")
        for hit in hits:
            print(hit)

In [16]:
from pymilvus import WeightedRanker

# 定义加权融合策略
ranker = WeightedRanker(0.5, 0.5)
# 测试
hybrid_search("教育的负面作用是什么", ranker)

TopK results:
{'id': 21, 'distance': 0.7465271949768066, 'entity': {'content': '### （六）生产力与教育的关系  \n生产力对其它一切因素都起着决定的作用，是决定教育性质的根本因素。  \n生产力对教育的主要作用表现为：生产力发展水平决定着教育事业发展的规模和速度；生产力发展水平制约着人才培养的规格与教育结构；生产力的发展促进教育内容、教学方法和教学组织形式的发展与改革。  \n教育对生产力的作用表现为：教育再生产劳动力；教育是科学知识与技术发展的重要手段。', 'h2': '第二节 教育和社会的关系'}}
{'id': 15, 'distance': 0.40837913751602173, 'entity': {'content': '# 第二章 教育基本原理  \n## 第一节 教育的功能  \n### （一）个体发展功能和社会发展功能  \n教育的正向功能（积极功能）指教育有助于社会进步和个体发展的积极影响和作用。  \n教育的负向功能（消极功能）指阻碍社会进步和个体发展的消极影响和作用。  \n教育的显性功能是指教育活动依照教育目的，在实际运行中所出现的与之相吻合的结果。  \n教育的隐性功能指伴随显性功能所出现的非预期性的功能。', 'h2': '第一节 教育的功能'}}


In [17]:
# 定义RRF融合策略
from pymilvus import RRFRanker

ranker = RRFRanker(k=60)
# 测试
hybrid_search("教育的负面作用是什么", ranker)

TopK results:
{'id': 21, 'distance': 0.03226646035909653, 'entity': {'content': '### （六）生产力与教育的关系  \n生产力对其它一切因素都起着决定的作用，是决定教育性质的根本因素。  \n生产力对教育的主要作用表现为：生产力发展水平决定着教育事业发展的规模和速度；生产力发展水平制约着人才培养的规格与教育结构；生产力的发展促进教育内容、教学方法和教学组织形式的发展与改革。  \n教育对生产力的作用表现为：教育再生产劳动力；教育是科学知识与技术发展的重要手段。', 'h2': '第二节 教育和社会的关系'}}
{'id': 15, 'distance': 0.016393441706895828, 'entity': {'content': '# 第二章 教育基本原理  \n## 第一节 教育的功能  \n### （一）个体发展功能和社会发展功能  \n教育的正向功能（积极功能）指教育有助于社会进步和个体发展的积极影响和作用。  \n教育的负向功能（消极功能）指阻碍社会进步和个体发展的消极影响和作用。  \n教育的显性功能是指教育活动依照教育目的，在实际运行中所出现的与之相吻合的结果。  \n教育的隐性功能指伴随显性功能所出现的非预期性的功能。', 'h2': '第一节 教育的功能'}}


In [18]:
# RRF 混合检索

def hybrid_search(query: str, ranker, filter_query: str | None = None):
    # 1.创建稠密、稀疏请求
    # 稠密
    query_dense_vector = embeddings.embed_query(query)
    dense_request = AnnSearchRequest(
        data=[query_dense_vector],
        anns_field="dense",
        param={"metric_type": "COSINE"},
        limit=3,
        filter=filter_query
    )
    # 稀疏
    sparse_request = AnnSearchRequest(
        data=[query],
        anns_field="sparse",
        param={"metric_type": "BM25"},
        limit=3,
        filter=filter_query
    )
    # 混合在一起
    reqs = [dense_request, sparse_request]

    # 2.设定融合策略，发送请求
    res = client.hybrid_search(
        collection_name=collection_name,
        reqs=reqs,
        ranker=ranker,
        limit=2,
        output_fields=['content', 'h2']
    )
    for hits in res:
        print("TopK results:")
        for hit in hits:
            print(hit)
   
            
from pymilvus import WeightedRanker

# 定义加权融合策略
ranker = WeightedRanker(0.5, 0.5)
# 测试
hybrid_search("教育的负面作用是什么", ranker, filter_query="h2 == '第一节 教育的功能'")

TopK results:
{'id': 15, 'distance': 0.7393761873245239, 'entity': {'content': '# 第二章 教育基本原理  \n## 第一节 教育的功能  \n### （一）个体发展功能和社会发展功能  \n教育的正向功能（积极功能）指教育有助于社会进步和个体发展的积极影响和作用。  \n教育的负向功能（消极功能）指阻碍社会进步和个体发展的消极影响和作用。  \n教育的显性功能是指教育活动依照教育目的，在实际运行中所出现的与之相吻合的结果。  \n教育的隐性功能指伴随显性功能所出现的非预期性的功能。', 'h2': '第一节 教育的功能'}}
